# Project 4 | NHS Data Engineering Portfolio | Yusuf Ismail
## NHS Cancer Waiting Times — Silver Transformation

**Notebook 02 of 5** — Silver layer. This is the heart of the pipeline.

My Silver layer turns the four faithful-but-raw Bronze files into one clean, consistent,
analysis-ready dataset. Every transformation here is deliberate and documented, because
every number in my Gold tables and my three articles will trace back to a decision I make
in this notebook.

**Input:** `data/bronze/*.parquet` (4 files, 1,076,865 rows total)
**Output:** clean per-standard Silver Parquet files in `data/silver/`

**What Silver does, in order:**
1. Combine all three Bronze years into one frame
2. Filter to **Provider** basis (avoids double-counting; most complete NHS assessment)
3. Filter to **October 2023 onwards** (post-framework, comparable scope)
4. Derive a proper `period_date` (real datetime) and clean `financial_year`
5. Normalise the casing inconsistencies I found in exploration
6. Recalculate compliance from `Within` / `Total` (I trust my own maths, not the supplied field)
7. Preserve nulls throughout — suppressed small numbers never become zeros
8. Write tidy per-standard Silver files for Gold to consume

**Principles:** every step is reversible in logic, documented in plain English, and verified
with an output before I move on. I never transform blind.

In [64]:
# I import pandas for the data and pathlib for portable paths.
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# I define my paths. I read from bronze, I write to silver.
PROJECT_DIR = Path.cwd().parent
BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
SILVER_DIR = PROJECT_DIR / "data" / "silver"

# I load all three Bronze Parquet files and stack them into one frame.
# I sort the file list so the years combine in chronological order.
bronze_files = sorted(BRONZE_DIR.glob("cwt_bronze_*.parquet"))
print("Bronze files I am combining:")
for f in bronze_files:
    print(f"    • {f.name}")

# pd.concat stacks them vertically. ignore_index=True gives one clean continuous index.
df = pd.concat([pd.read_parquet(f) for f in bronze_files], ignore_index=True)

print(f"\nCombined Bronze frame: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Expected: 1,076,865 rows × 33 columns")

Bronze files I am combining:
    • cwt_bronze_2022-23.parquet
    • cwt_bronze_2023-24.parquet
    • cwt_bronze_2024-25.parquet
    • cwt_bronze_2025-26-Apr-Sep.parquet

Combined Bronze frame: 1,076,865 rows × 33 columns
Expected: 1,076,865 rows × 33 columns


## Step 1 — Filter to Provider basis and October 2023 onwards

I apply two filters in sequence and watch the row count drop at each step, so I can prove
exactly what I removed and why.

**Filter 1 — Provider basis.** The file holds both Provider and Commissioner rows. They
measure the same patients from different angles, so keeping both would double-count. NHS
England states provider-based data is the most complete assessment of performance, so I keep
Provider and drop Commissioner.

**Filter 2 — October 2023 onwards.** The cancer standards framework changed in October 2023.
Data before that is not directly comparable for the 31D and 62D standards. I scope to
Oct 2023 → Sep 2025 for clean, comparable analysis across all standards.

In [66]:
# I track the row count at every filter step so I have a transparent audit trail.
print(f"Starting rows: {len(df):,}")

# Filter 1: Provider basis only.
df = df[df["Basis"] == "Provider"].copy()
print(f"After keeping Provider only: {len(df):,}")

# I convert Period to a real datetime BEFORE filtering. My source files use two different
# date formats: the older files use ISO 'YYYY-MM-DD' (2024-11-01) and the newer Apr-Sep 2025
# file uses UK 'DD/MM/YYYY' (01/04/2025). String comparison silently drops the UK-format rows,
# so I parse to a real datetime first and filter on that. format='mixed' lets pandas handle
# both layouts; dayfirst=True tells it that ambiguous dates like 01/04/2025 are day-first.
df["period_date"] = pd.to_datetime(df["Period"], format="mixed", dayfirst=True)

# Filter 2: October 2023 onwards — now a proper datetime comparison, format-independent.
df = df[df["period_date"] >= "2023-10-01"].copy()
print(f"After keeping Oct 2023 onwards: {len(df):,}")

print(f"\nMy analysis frame is now {len(df):,} rows.")

Starting rows: 1,076,865
After keeping Provider only: 765,412
After keeping Oct 2023 onwards: 460,238

My analysis frame is now 460,238 rows.


## Step 2 — Derive clean time columns

The `Period` column is still a string (e.g. '2024-11-01'). I convert it to a real datetime
so I can do proper time-series work in Gold — sorting, month arithmetic, clean axis labels.

I derive three time columns:
- `period_date` — the real datetime (first of the reporting month)
- `period_month` — a readable 'YYYY-MM' label for grouping and charts
- `financial_year` — already in Bronze as `_financial_year`, but I surface it cleanly here

I keep the original `Period` string untouched as well, so I never lose the raw value.

In [68]:
# I convert the Period string into a real datetime. NHS uses the first of the month
# to represent the whole reporting month, so '2024-11-01' means "November 2024".
# period_date is already created in Step 1 (parsed from mixed date formats before filtering).
# Here I derive the readable month label and surface the financial year.
df["period_month"] = df["period_date"].dt.strftime("%Y-%m")

# I surface the financial year cleanly from my Bronze audit column.
df["financial_year"] = df["_financial_year"]

# I verify the conversion worked and the date range matches my scope.
print("period_date dtype:", df["period_date"].dtype)
print("Date range:", df["period_date"].min().date(), "to", df["period_date"].max().date())
print("Distinct months:", df["period_month"].nunique())
print()
print("Months present:")
print(sorted(df["period_month"].unique()))

period_date dtype: datetime64[ns]
Date range: 2023-10-01 to 2025-09-01
Distinct months: 24

Months present:
['2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09']


## Step 3 — Normalise the casing inconsistency

In exploration I found the same referral routes written in different cases. Looking closer,
the split is systematic: the FDS standard uses ALL-CAPS route names while the 62D standard
uses Title Case for the same real-world routes.

| Raw (FDS)                                  | Raw (62D)                 | My canonical label        |
|--------------------------------------------|---------------------------|---------------------------|
| URGENT SUSPECTED CANCER                     | Urgent Suspected Cancer   | Urgent Suspected Cancer   |
| NATIONAL SCREENING PROGRAMME                | Screening                 | Screening                 |
| BREAST SYMPTOMATIC, CANCER NOT SUSPECTED    | Breast Symptomatic        | Breast Symptomatic        |

I create a new column `referral_route_clean` with one canonical label per route. I keep the
original `Referral_Route_or_Stage` untouched so nothing is lost. This protects me if I ever
compare routes across standards, and it makes my Gold route analysis unambiguous.

Note: `ALL ROUTES`, `ALL STAGES`, `First Treatment`, `Subsequent`, and `Consultant Upgrade`
are already consistent and need no remapping — I leave them as-is in the clean column.

In [70]:
# I build an explicit mapping from every raw route variant to one canonical label.
# Explicit is better than clever here — anyone reading this sees exactly what maps to what.
route_map = {
    "URGENT SUSPECTED CANCER": "Urgent Suspected Cancer",
    "Urgent Suspected Cancer": "Urgent Suspected Cancer",
    "NATIONAL SCREENING PROGRAMME": "Screening",
    "Screening": "Screening",
    "BREAST SYMPTOMATIC, CANCER NOT SUSPECTED": "Breast Symptomatic",
    "Breast Symptomatic": "Breast Symptomatic",
}

# I create the clean column. For any value not in my map (ALL ROUTES, Consultant Upgrade,
# First Treatment, etc.) I keep the original value unchanged — they are already consistent.
df["referral_route_clean"] = df["Referral_Route_or_Stage"].map(route_map).fillna(df["Referral_Route_or_Stage"])

# I verify: the clean column should have fewer distinct values than the raw one,
# because I have collapsed the casing duplicates.
print("Raw distinct routes:  ", df["Referral_Route_or_Stage"].nunique())
print("Clean distinct routes:", df["referral_route_clean"].nunique())
print()
print("Clean route values:")
for v in sorted(df["referral_route_clean"].dropna().unique()):
    print(f"    • {v}")

Raw distinct routes:   12
Clean distinct routes: 9

Clean route values:
    • ALL ROUTES
    • ALL STAGES
    • Breast Symptomatic
    • Consultant Upgrade
    • Exhibited (non-cancer) breast symptoms - cancer not initially suspected
    • First Treatment
    • Screening
    • Subsequent
    • Urgent Suspected Cancer


## Step 4 — Recalculate compliance and own the maths

NHS England supplies a `Performance` column, but I recalculate compliance myself from
`Within` / `Total`. I verified my calculation matches their supplied figure to within
floating-point rounding error (max difference ~5e-16), so I am not correcting them —
I am taking ownership, so I can explain the precision and the null-handling behind every
percentage in my articles.

I create `compliance_rate` = `Within` / `Total`, with two deliberate rules:
- If `Total` is 0 or null, `compliance_rate` is null (I never divide by zero or invent a rate).
- The two referral items (USC, UBS) have no Within/After split, so their `compliance_rate`
  is null by design — they measure referral activity, not a pass/fail standard.

I also derive `breach_count` = `After`, surfaced under a clear analytical name, and
`breach_rate` = `After` / `Total` for convenience in Gold.

In [72]:
# I recalculate compliance from the source counts. I guard against divide-by-zero:
# where Total is null or zero, the result is null (not an error, not a fake number).
import numpy as np

df["compliance_rate"] = np.where(
    (df["Total"].notna()) & (df["Total"] > 0),
    df["Within"] / df["Total"],
    np.nan
)

# I surface the breach count under a clear analytical name, and a breach rate for convenience.
df["breach_count"] = df["After"]
df["breach_rate"] = np.where(
    (df["Total"].notna()) & (df["Total"] > 0),
    df["After"] / df["Total"],
    np.nan
)

# I verify my recalculation matches NHS's supplied Performance where both exist.
check = df[df["Performance"].notna() & df["compliance_rate"].notna()].copy()
max_diff = (check["Performance"] - check["compliance_rate"]).abs().max()
print(f"Rows checked against supplied Performance: {len(check):,}")
print(f"Max difference between my calc and theirs: {max_diff:.2e}")
print(f"My calc matches NHS to rounding error: {max_diff < 1e-9}")

# I confirm where my compliance_rate is null, and that it is null for the right reasons.
print(f"\ncompliance_rate null rows: {df['compliance_rate'].isna().sum():,}")
print("Null compliance_rate by standard (expected: the two referral items):")
print(df[df["compliance_rate"].isna()]["Standard_or_Item"].value_counts())

Rows checked against supplied Performance: 417,263
Max difference between my calc and theirs: 5.00e-10
My calc matches NHS to rounding error: True

compliance_rate null rows: 42,975
Null compliance_rate by standard (expected: the two referral items):
Standard_or_Item
Urgent Suspected Cancer referral      40337
Urgent Breast Symptomatic referral     2638
Name: count, dtype: int64


## Step 5 — Define the join key, then split into headline + bands

I am splitting Silver into two clean entities:
- **Primary Silver** — 22 coherent columns. Every column means one thing. This is what I
  load for almost all my analysis.
- **Silver bands** — the 17 waiting-time band columns, cleaned to the same standard, plus
  the join key. I load this only when I want severity detail (e.g. "patients waiting >104 days").

For the split to be lossless, I need a **join key**: a set of columns that uniquely identifies
every row, so I can rejoin bands to headline unambiguously. My natural key is the full grain:
`period_date + Org_Code + Standard_or_Item + Cancer_Type + Referral_Route_or_Stage + Treatment_Modality`.

Before I split anything, I prove this key is unique — no two rows share the same combination.
If it is not unique, the split is unsafe and I fix it before proceeding.

In [74]:
# I define my candidate join key — the full dimensional grain of a row.
join_key = [
    "period_date", "Org_Code", "Standard_or_Item",
    "Cancer_Type", "Referral_Route_or_Stage", "Treatment_Modality"
]

# I test uniqueness: the number of unique key combinations must equal the number of rows.
n_rows = len(df)
n_unique = df.groupby(join_key, dropna=False).ngroups

print(f"Total rows:              {n_rows:,}")
print(f"Unique key combinations: {n_unique:,}")
print(f"Key is unique:           {n_rows == n_unique}")

# If not unique, I want to SEE the offending duplicates rather than guess.
if n_rows != n_unique:
    dupes = df[df.duplicated(subset=join_key, keep=False)].sort_values(join_key)
    print(f"\n{len(dupes):,} rows are involved in duplicate keys. Sample:")
    print(dupes[join_key].head(10).to_string(index=False))

Total rows:              460,238
Unique key combinations: 460,238
Key is unique:           True


## Step 6 — Write the Silver files

I write two sets of files, both split by standard so Gold consumes focused inputs:

**Primary Silver (headline measures):**
- `silver_fds.parquet`
- `silver_31d.parquet`
- `silver_62d.parquet`
- `silver_referrals.parquet` (USC + UBS combined — they share the referral-activity grain)

**Silver bands (severity detail, same rows, joinable on my proven key):**
- `silver_bands_fds.parquet`
- `silver_bands_31d.parquet`
- `silver_bands_62d.parquet`
- `silver_bands_referrals.parquet`

I keep nulls intact throughout. I write a log so I have a receipt of every file, its rows,
and its columns.

In [76]:
# I define exactly which columns belong to my clean primary Silver.
headline_cols = [
    # identity & time
    "period_date", "period_month", "financial_year",
    "Org_Code", "Org_Name", "Parent_Org",
    # dimensions
    "Standard_or_Item", "Cancer_Type",
    "Referral_Route_or_Stage", "referral_route_clean", "Treatment_Modality",
    # headline measures
    "Total", "Within", "After", "Performance",
    "compliance_rate", "breach_count", "breach_rate",
    # audit trail
    "_source_file", "_ingested_at",
]

# I define the 17 band columns that go into the separate bands file.
band_cols = [
    "Within_14_days", "In_15_to_28_days", "In_29_to_42_days", "In_43_to_62_days", "After_62_days",
    "Within_31_days", "In_32_to_38_days", "In_39_to_48_days", "In_49_to_62_days",
    "In_63_to_76_days", "In_77_to_90_days", "In_91_to_104_days", "After_104_days",
    "In_15_to_16_days", "In_17_to_21_days", "In_22_to_28_days", "After_28_days",
]

# The bands file = the join key + the band columns. That is all it needs to be joinable and useful.
bands_file_cols = join_key + band_cols

print("Primary Silver columns:", len(headline_cols))
print("Bands file columns:", len(bands_file_cols), f"(key {len(join_key)} + bands {len(band_cols)})")

# I sanity-check every column I named actually exists in my frame, so I fail loudly now
# rather than silently later.
missing_headline = [c for c in headline_cols if c not in df.columns]
missing_bands = [c for c in bands_file_cols if c not in df.columns]
print("\nMissing headline columns (expect none):", missing_headline)
print("Missing bands columns (expect none):", missing_bands)

Primary Silver columns: 20
Bands file columns: 23 (key 6 + bands 17)

Missing headline columns (expect none): []
Missing bands columns (expect none): []


In [77]:
# I map each standard to its output filename stem. USC and UBS share the referrals file
# because they have the same referral-activity grain.
standard_groups = {
    "fds":       ["FDS"],
    "31d":       ["31D"],
    "62d":       ["62D"],
    "referrals": ["Urgent Suspected Cancer referral", "Urgent Breast Symptomatic referral"],
}

silver_log = []

for stem, standards in standard_groups.items():
    # I select the rows for this standard group.
    subset = df[df["Standard_or_Item"].isin(standards)]

    # I write the clean primary Silver file (headline columns only).
    primary_path = SILVER_DIR / f"silver_{stem}.parquet"
    subset[headline_cols].to_parquet(primary_path, index=False)

    # I write the parallel bands file (key + band columns), joinable to the primary on my proven key.
    bands_path = SILVER_DIR / f"silver_bands_{stem}.parquet"
    subset[bands_file_cols].to_parquet(bands_path, index=False)

    # I record receipts for both files.
    silver_log.append({"file": primary_path.name, "rows": len(subset), "cols": len(headline_cols)})
    silver_log.append({"file": bands_path.name, "rows": len(subset), "cols": len(bands_file_cols)})

    print(f"{stem}: wrote {len(subset):,} rows -> {primary_path.name} + {bands_path.name}")

print("\n=== Silver write log ===")
print(pd.DataFrame(silver_log).to_string(index=False))

fds: wrote 91,801 rows -> silver_fds.parquet + silver_bands_fds.parquet
31d: wrote 144,495 rows -> silver_31d.parquet + silver_bands_31d.parquet
62d: wrote 180,967 rows -> silver_62d.parquet + silver_bands_62d.parquet
referrals: wrote 42,975 rows -> silver_referrals.parquet + silver_bands_referrals.parquet

=== Silver write log ===
                          file   rows  cols
            silver_fds.parquet  91801    20
      silver_bands_fds.parquet  91801    23
            silver_31d.parquet 144495    20
      silver_bands_31d.parquet 144495    23
            silver_62d.parquet 180967    20
      silver_bands_62d.parquet 180967    23
      silver_referrals.parquet  42975    20
silver_bands_referrals.parquet  42975    23


## Step 7 — Verify the split is lossless (rejoin test)

This is the test that validates my entire split decision. I reload the primary 62D file and
its bands file, then join them back together on my proven key. If the split is sound:
1. The join produces exactly the same number of rows as the primary file (no duplicates, no drops)
2. Every row finds its match (no nulls introduced by the join)
3. A spot-checked severity number (patients waiting >104 days) is recoverable

If this passes, I can trust that any time I need band detail, I can reattach it cleanly.

In [93]:
# I reload the two 62D files as if I were starting fresh in a Gold notebook.
primary_62d = pd.read_parquet(SILVER_DIR / "silver_62d.parquet")
bands_62d = pd.read_parquet(SILVER_DIR / "silver_bands_62d.parquet")

print(f"Primary 62D rows: {len(primary_62d):,}")
print(f"Bands 62D rows:   {len(bands_62d):,}")

# I join them back on the proven key. A left join from primary should match every row exactly once.
rejoined = primary_62d.merge(bands_62d, on=join_key, how="left", validate="one_to_one")

print(f"\nRejoined rows:    {len(rejoined):,}")
print(f"Join is lossless (same row count): {len(rejoined) == len(primary_62d)}")

# validate='one_to_one' would have raised an error if the key wasn't unique on both sides,
# so reaching this line already proves the join integrity. I confirm explicitly too.
print("Join validated as one-to-one: True (merge would have errored otherwise)")

# I spot-check a real severity number: how many 62D patients waited more than 104 days,
# nationally, in the most recent month — a number I could never get from the headline file alone.
recent = rejoined[
    (rejoined["Org_Code"] == "Total") &
    (rejoined["Cancer_Type"] == "ALL CANCERS") &
    (rejoined["referral_route_clean"] == "Urgent Suspected Cancer") &
    (rejoined["Treatment_Modality"] == "ALL MODALITIES") &
    (rejoined["period_month"] == "2025-09")
]
if len(recent):
    waited_104 = recent["After_104_days"].iloc[0]
    total = recent["Total"].iloc[0]
    print(f"\nSeverity spot-check — 62D, Urgent Suspected Cancer, England, 2025-09:")
    print(f"   Patients who waited MORE than 104 days: {waited_104:,.0f} out of {total:,.0f}")

Primary 62D rows: 180,967
Bands 62D rows:   180,967

Rejoined rows:    180,967
Join is lossless (same row count): True
Join validated as one-to-one: True (merge would have errored otherwise)

Severity spot-check — 62D, Urgent Suspected Cancer, England, 2025-09:
   Patients who waited MORE than 104 days: 1,801 out of 17,392
